<a href="https://colab.research.google.com/github/memo124/Tareas---KODIGO/blob/main/welcome_to_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Paso 0: Agregar importaciones de librerias

In [ ]:
from abc import ABC, abstractmethod
from pathlib import Path
import logging

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
FILE_PATH = "/content/exercises.csv"

## Paso 1: Clase Abstracta

In [ ]:
class DataComponent(ABC):
  @abstractmethod
  def execute(self):
    pass

## Paso 2: Mixins (Logger y Validator)

In [ ]:
logging.basicConfig(
    level = logging.INFO,
    format = "[%(asctime)s] [%(levelname)s - %(name)s] %(message)s"
)

class LoggerMixin:
    _logger: logging.Logger | None = None
    
    @property
    def logger(self) -> logging.Logger:
        return self._logger or logging.getLogger(type(self).__name__)
    
    def log_info(self, message):
        self.logger.info(message)

    def log_warning(self, message):
        self.logger.warning(message)

    def log_error(self, message):
        self.logger.error(message)
        
class ValidatorMixin:
    
    def validate_file(self, file_path):
        # Verify that the path corresponds to an existing file
        if not Path(file_path).is_file():
            raise FileNotFoundError(
                f"No existe el archivo: {file_path}"
            )
            
    def validate_columns(self, dataframe, required_columns):
        # Find the required columns that do not exist in the DataFrame
        missing_columns = [
            column
            for column in required_columns
            if column not in dataframe.columns
        ]

        # Stops the process if at least one column is missing
        if missing_columns:
            raise ValueError(
                f"Faltan las columnas: {missing_columns}"
            )
            
    def validate_nulls(self, dataframe, columns=None):
        columns = columns or dataframe.columns
        
        null_counts = dataframe[columns].isna().sum()
        columns_with_nulls = null_counts[null_counts > 0]

        # Stops the process if null values ​​are found
        if not columns_with_nulls.empty:
            raise ValueError(
                f"Columnas con valores nulos:\n{columns_with_nulls}"
            )

## Paso 3: Clase DataLoader

In [ ]:
class DataLoader(DataComponent, ValidatorMixin, LoggerMixin):
    def __init__(self, file_path):
        self.__file_path = file_path
        
    def execute(self):
        try:
            self.log_info(f"Cargando archivo: {self.__file_path}")
            # Validate that the file exists before reading it.
            self.validate_file(self.__file_path)
        
            # Load the CSV file into a DataFrame.
            dataframe = pd.read_csv(self.__file_path)
            self.log_info(f"Archivo cargado correctamente: {len(dataframe)} filas")
            return dataframe
        except Exception as error:
            self.log_error(f"Error al cargar los datos: {error}")
            raise

## Paso 4: Clase DataCleaner

In [ ]:
class DataCleaner(DataComponent, LoggerMixin):
    # Cleaning component: infers types and imputes missing values.
    def __init__(self, dataframe, categorical_fill=None):
        # Stores the received data and optional cleaning settings.
        self.__dataframe = dataframe
        self.__categorical_fill = categorical_fill

    @property
    def summary(self):
        # Read-only descriptive statistics of the dataset.
        return self.__dataframe.describe(include="all")

    @property
    def null_report(self):
        # Read-only null count per column.
        return self.__dataframe.isna().sum()

    def execute(self):
        # Runs the full cleaning flow and returns the clean DataFrame.
        try:
            self.log_info("Iniciando limpieza de datos")
            self.__dataframe = self.__dataframe.copy()
            self.__convert_types()
            self.__impute_nulls()
            self.log_info("Limpieza de datos completada")
            return self.__dataframe
        except Exception as error:
            self.log_error(f"Error durante la limpieza de datos: {error}")
            raise

    def __convert_types(self):
        # Applies Pandas automatic type conversion.
        self.log_info("Convirtiendo tipos de datos")
        self.__dataframe = self.__dataframe.convert_dtypes()

    def __impute_nulls(self):
        # Fills missing values: median, configured value, or mode.
        self.log_info("Imputando valores nulos")
        for column in self.__dataframe.columns:
            null_count = self.__dataframe[column].isna().sum()

            # Skip columns without missing values.
            if null_count == 0:
                continue
            is_numeric = pd.api.types.is_numeric_dtype(self.__dataframe[column])
            mode_values = self.__dataframe[column].mode()

            # Numeric columns use the median; text columns use a configured value or mode.
            if is_numeric:
                fill_value = self.__dataframe[column].median()
            elif self.__categorical_fill is not None:
                fill_value = self.__categorical_fill
            else:
                fill_value = mode_values[0] if not mode_values.empty else "unknown"

            self.__dataframe[column] = self.__dataframe[column].fillna(fill_value)
            self.log_warning(f"Columna '{column}': {null_count} valores nulos imputados")

## Paso 5: DataAnalyst

In [ ]:
class DataAnalyst(DataComponent, LoggerMixin):
    # Allowed report types: console, graphic, or both.
    def __init__(self, dataframe, report_type="both", graphic_columns=None):
        self.__dataframe = dataframe
        self.__report_type = report_type
        self.__graphic_columns = graphic_columns


    def execute(self):
        self.log_info("Analizando los datos")
        
        try:
            if self.__report_type not in ["console", "graphic", "both"]:
                raise ValueError("Tipo de reporte no valido")

            # Generates the statistical summary in the console when requested.
            if self.__report_type in ["console", "both"]:
                print(self.__dataframe.describe())
                
            # Generates charts when the graphic report is requested.
            if self.__report_type in ["graphic", "both"]:
                # Uses numeric columns by default when no columns are selected.
                if self.__graphic_columns is None:
                    self.__dataframe.hist(figsize=(10, 6))
                else:
                    # Creates one chart for each selected column.
                    for column in self.__graphic_columns:
                        self.__dataframe[column].value_counts().plot(kind="bar")
                        plt.title(column)
                        plt.show()

                if self.__graphic_columns is None:
                    plt.show()
                    
        except Exception as error:
            self.log_error(error)
            raise

## Paso 6: Main del Notebook

In [ ]:
report_type = "both" #@param ["console", "graphic", "both"]
graphic_columns = ["category", "difficulty", "equipment"]

def create_loader(data):
    return DataLoader(FILE_PATH)

def create_cleaner(data):
    return DataCleaner(data, categorical_fill="unknown")

def create_analyst(data):
    return DataAnalyst(data, report_type, graphic_columns)

In [ ]:
try:
    # Pipeline steps executed in order.
    pipeline = [create_loader, create_cleaner, create_analyst]
    # Initial input for the first component.
    data = None

    for create_component in pipeline:
        component = create_component(data)
        # Polymorphism: each component runs with execute().
        result = component.execute()

        if result is not None:
            data = result
except Exception as error:
    print(f"Pipeline detenido: {error}")